# Preparar los datos de la Sumativa 1

Este notebook prepara una sola copia de trabajo. Se ejecuta una vez por equipo, después de descargar el original. Después se trabaja con `02_Sumativa1.ipynb`.

La guía de datasets, página 3, pide ocultar al azar entre el 10% y el 15% de una o dos variables numéricas cuando el original no tiene faltantes. Ocultaremos el 10% del tiempo en productos, con semilla 42. El original nunca se modificará. No rellenamos valores.

Abrir este archivo dentro de `Sumativa_1` con el kernel **Python (Estadistica Computacional)**. La carpeta `datos_trabajo` ya forma parte del proyecto.

## 1 Leer el original

`Path` permite comprobar si un archivo existe. `float_precision="round_trip"` conserva los decimales del archivo original. Primero comprobamos su tamaño y sus faltantes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

original = pd.read_csv("datos_originales/online_shoppers_intention.csv", float_precision="round_trip")
print("Filas y columnas:", original.shape)
print("Faltantes originales:", original.isna().sum().sum())

Filas y columnas: (12330, 18)
Faltantes originales: 0


## 2 Crear la copia solo si aún no existe

Si ambos archivos ya existen, los reutilizamos. Si solo existe uno, nos detenemos para revisar la preparación en lugar de sobrescribirla.

En un equipo nuevo: `default_rng(42)` fija el generador aleatorio; `choice()` elige posiciones sin repetir (`replace=False`); `np.sort()` las ordena. No elegimos por tiempo ni por compra: cada fila tiene la misma probabilidad, que es la idea de **MCAR**.

`np.nan` marca un dato faltante. Antes de ocultarlo, guardamos su posición y su valor en otro CSV. La posición empieza en cero y no es un identificador de cliente. `lineterminator="\n"` hace que se guarde un salto de línea uniforme en ambos equipos.

In [2]:
archivo_trabajo = Path("datos_trabajo/online_shoppers_mcar.csv")
archivo_respaldo = Path("datos_trabajo/valores_ocultados.csv")

if archivo_trabajo.exists() and archivo_respaldo.exists():
    print("Usamos los archivos existentes. No se vuelve a sortear ni a guardar.")
elif archivo_trabajo.exists() or archivo_respaldo.exists():
    raise ValueError("Falta uno de los dos archivos. Revisar la preparación antes de continuar.")
else:
    cantidad = int(len(original) * 0.10)
    generador = np.random.default_rng(42)
    posiciones = generador.choice(len(original), size=cantidad, replace=False)
    posiciones = np.sort(posiciones)

    respaldo = pd.DataFrame()
    respaldo["fila_original"] = posiciones
    respaldo["valor_original"] = original.loc[posiciones, "ProductRelated_Duration"].to_numpy()

    trabajo = original.copy()
    trabajo.loc[posiciones, "ProductRelated_Duration"] = np.nan
    trabajo.to_csv(archivo_trabajo, index=False, lineterminator="\n")
    respaldo.to_csv(archivo_respaldo, index=False, lineterminator="\n")
    print("Datos preparados. Tiempos ocultados:", cantidad)

Usamos los archivos existentes. No se vuelve a sortear ni a guardar.


## 3 Revisar la copia guardada

Esperamos 12.330 filas, 18 columnas y 1.233 tiempos faltantes. El respaldo permite conocer los valores originales en etapas futuras del proyecto. Una vez creada esta versión, no borrarla para generar otra.

In [3]:
trabajo = pd.read_csv(archivo_trabajo, float_precision="round_trip")
respaldo = pd.read_csv(archivo_respaldo, float_precision="round_trip")
print("Filas y columnas:", trabajo.shape)
print("Tiempos faltantes:", trabajo["ProductRelated_Duration"].isna().sum())
print("Valores conservados en el respaldo:", len(respaldo))
respaldo.head()

Filas y columnas: (12330, 18)
Tiempos faltantes: 1233
Valores conservados en el respaldo: 1233


,fila_original,valor_original
0,50,0.000000
1,56,0.000000
2,62,1553.583333
3,63,706.500000
4,77,76.000000


La preparación termina aquí. Abrir `02_Sumativa1.ipynb` para el análisis y ejecutar sus celdas en orden. Las instrucciones de descarga y la comprobación opcional del archivo están en los README de datos.